# 07. Reports and dashboards

Forecasts usually end up in front of other people. This notebook turns a result into a one-page report, an interactive dashboard, plain charts for slides and files other tools can read, and shows how to save, reload and quickly refit a model.

- One-page report
- Interactive dashboard
- Interactive charts on their own
- Plain charts for slides
- Saving and loading a model
- Refitting quickly with a warm start
- Exporting the forecast

Previous: [06. Evaluation and tuning](06_evaluation_and_tuning.ipynb) | Next: [08. Many series on Apple Silicon](08_many_series_mlx.ipynb)

In [ ]:
import tempfile
import time
from pathlib import Path

from prophet import Prophet, forecast
from prophet.plot import plot_components_plotly, plot_plotly
from prophet.serialize import model_from_json, model_to_json
from prophet.utilities import warm_start_params

from helpers import load_example, quiet_logs

quiet_logs()
out = Path(tempfile.mkdtemp(prefix="neoprophet_"))  # every file below goes here, not into the repo
df = load_example("peyton_manning")
result = forecast(df, horizon="12 months", freq="D", country_holidays="US")

## One-page report

`result.report()` lays out the forecast, the key numbers, the components and backtest accuracy on one A4 page. It runs the backtest itself; pass `backtest=False` to skip it, or an existing backtest to reuse it.

In [ ]:
bt = result.backtest()
fig = result.report(backtest=bt)

In [ ]:
fig.savefig(out / "forecast-report.pdf")
fig.savefig(out / "forecast-report.png", dpi=150)
sorted(p.name for p in out.iterdir())

## Interactive dashboard

`result.dashboard(path)` writes a single HTML file with zoom, hover and the same numbers as the report. Plotly's JavaScript is embedded, so it opens offline and can be emailed as one file. It needs `plotly` installed.

In [ ]:
path = result.dashboard(out / "dashboard.html", backtest=bt)
print(path, f"{path.stat().st_size / 1e6:.1f} MB")

## Interactive charts on their own

`plot_plotly` and `plot_components_plotly` return plotly figures in the same style. Showing them inline in Jupyter also needs the `nbformat` package; here they are written to HTML files instead, which works everywhere.

In [ ]:
m, fcst = result.model, result.predictions
plot_plotly(m, fcst).write_html(out / "forecast.html", include_plotlyjs="cdn")
plot_components_plotly(m, fcst).write_html(out / "components.html", include_plotlyjs="cdn")
sorted(p.name for p in out.glob("*.html"))

## Plain charts for slides

The automatic title, subtitle, callouts and credit line are useful in a notebook but can crowd a slide that already has a heading. Each can be switched off.

In [ ]:
fig = result.plot(title=False, annotate=False, source=False)

## Saving and loading a model

`model_to_json` stores a fitted model as JSON, including its parameters and history, so it can be reloaded later or elsewhere without refitting. Avoid pickle: JSON is safer to load and survives library upgrades better.

In [ ]:
(out / "model.json").write_text(model_to_json(result.model))
reloaded = model_from_json((out / "model.json").read_text())
future = reloaded.make_future_dataframe(periods=30)
reloaded.predict(future)[["ds", "yhat"]].tail(3)

## Refitting quickly with a warm start

When new data arrives, a refit can start from the previous model's parameters instead of from scratch. The optimiser then has less work to do, which adds up when you refit many models every day.

In [ ]:
older, newer = df.iloc[:-30], df

start = time.perf_counter()
m_old = Prophet().fit(older)
cold = time.perf_counter() - start

start = time.perf_counter()
Prophet().fit(newer)
fresh = time.perf_counter() - start

start = time.perf_counter()
Prophet().fit(newer, init=warm_start_params(m_old))
warm = time.perf_counter() - start

print(f"first fit {cold:.2f} s | refit from scratch {fresh:.2f} s | warm-started refit {warm:.2f} s")

The saving is modest for one series and grows with the number of models. Keep the model configuration the same between fits; a warm start only makes sense when the parameters mean the same thing.

## Exporting the forecast

`result.forecast` holds the future rows with the interval; `result.predictions` holds every component for history and future. Both are ordinary DataFrames.

In [ ]:
result.forecast.to_csv(out / "forecast.csv", index=False)
result.forecast.head(3)

## Next

[08. Many series on Apple Silicon](08_many_series_mlx.ipynb) fits hundreds of series at once with the experimental MLX backend.